<a href="https://colab.research.google.com/github/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/blob/main/notebooks/final_model_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Kaggle download
import os
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
os.environ['KAGGLE_KEY'] = getpass('Kaggle API key: ')

!kaggle datasets download -d mlg-ulb/creditcardfraud -p /content/data/ulb --unzip
!kaggle datasets download -d kartik2112/fraud-detection -p /content/data/sparkov --unzip


Kaggle username: Subbareddy_subbz
Kaggle API key: ··········
Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
100% 66.0M/66.0M [00:00<00:00, 159MB/s]

Dataset URL: https://www.kaggle.com/datasets/kartik2112/fraud-detection
License(s): CC0-1.0
100% 202M/202M [00:02<00:00, 89.6MB/s]



In [2]:
# 2. Clone the repo (already has split.py and utils.py inside it)
github_token = getpass('Paste your GitHub token: ')
repo_url = f'https://{github_token}@github.com/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-.git'
!git clone {repo_url}


Paste your GitHub token: ··········
Cloning into 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 147 (delta 91), reused 18 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 417.97 KiB | 7.89 MiB/s, done.
Resolving deltas: 100% (91/91), done.


In [3]:

# 3. Move in, confirm, move data in
%cd Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-
!mv /content/data/ulb /content/data/sparkov data/


/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-


In [4]:
!pip install -q catboost imbalanced-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00


In [5]:
import sys
import numpy as np
import pandas as pd
sys.path.append('/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-')
from src.split import split_sparkov
from src.utils import fit_category_risk, apply_category_risk, add_prev_txn_features

In [6]:
sparkov_train, sparkov_test = split_sparkov()

# category risk: fit on train only, apply to both splits
category_risk_map = fit_category_risk(sparkov_train, 'category', 'is_fraud')
sparkov_train['category_risk'] = apply_category_risk(sparkov_train, 'category', category_risk_map)
sparkov_test['category_risk']  = apply_category_risk(sparkov_test, 'category', category_risk_map)

# mark which split each row came from, since train/test indexes overlap
sparkov_train['_split'] = 'train'
sparkov_test['_split']  = 'test'

# implied velocity: built on the full train+test timeline so boundary
# transactions still have a real previous transaction to compare against
full = pd.concat([sparkov_train, sparkov_test]).sort_values('trans_date_trans_time')
full = add_prev_txn_features(full, 'cc_num', 'trans_date_trans_time', 'merch_lat', 'merch_long')

# split back using the marker column, not the index
sparkov_train = full[full['_split'] == 'train'].drop(columns='_split')
sparkov_test  = full[full['_split'] == 'test'].drop(columns='_split')

print(sparkov_train.shape, sparkov_test.shape)

Train: 1,296,675 rows, from 2019-01-01 00:00:18 to 2020-06-21 12:13:37
Test:  555,719 rows, from 2020-06-21 12:14:25 to 2020-12-31 23:59:34
(1296675, 27) (555719, 27)


In [7]:
# velocity has extreme outliers, so log-transform it
# (first-ever transaction for a card has nothing to compare to -> 0)
sparkov_train['implied_velocity_kmh'] = sparkov_train['implied_velocity_kmh'].fillna(0)
sparkov_test['implied_velocity_kmh']  = sparkov_test['implied_velocity_kmh'].fillna(0)

sparkov_train['velocity_log'] = np.log1p(sparkov_train['implied_velocity_kmh'])
sparkov_test['velocity_log']  = np.log1p(sparkov_test['implied_velocity_kmh'])

# is_night: 10pm-3am, from the EDA
sparkov_train['hour'] = pd.to_datetime(sparkov_train['trans_date_trans_time']).dt.hour
sparkov_test['hour']  = pd.to_datetime(sparkov_test['trans_date_trans_time']).dt.hour
sparkov_train['is_night'] = sparkov_train['hour'].isin([22, 23, 0, 1, 2, 3]).astype(int)
sparkov_test['is_night']  = sparkov_test['hour'].isin([22, 23, 0, 1, 2, 3]).astype(int)

# amt_log: raw amount is heavily skewed
sparkov_train['amt_log'] = np.log1p(sparkov_train['amt'])
sparkov_test['amt_log']  = np.log1p(sparkov_test['amt'])

# age in years, from dob
sparkov_train['age'] = (pd.to_datetime(sparkov_train['trans_date_trans_time']) - pd.to_datetime(sparkov_train['dob'])).dt.days / 365.25
sparkov_test['age']  = (pd.to_datetime(sparkov_test['trans_date_trans_time'])  - pd.to_datetime(sparkov_test['dob'])).dt.days / 365.25

# gender_flag: 1 = male, 0 = female
sparkov_train['gender_flag'] = (sparkov_train['gender'] == 'M').astype(int)
sparkov_test['gender_flag']  = (sparkov_test['gender'] == 'M').astype(int)

# final six-feature matrices
features = ['amt_log', 'category_risk', 'is_night', 'velocity_log', 'age', 'gender_flag']
target = 'is_fraud'

X_train, y_train = sparkov_train[features], sparkov_train[target]
X_test, y_test = sparkov_test[features], sparkov_test[target]

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(1296675, 6) (555719, 6)
0.005788651743883394 0.0038598644278853163


In [8]:
from sklearn.model_selection import train_test_split

# carve a validation set out of train, for threshold tuning later
# stratify to keep the (rare) fraud rate consistent across both pieces
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

print(X_train_sub.shape, X_val.shape, X_test.shape)
print(y_train_sub.mean(), y_val.mean(), y_test.mean())

(1037340, 6) (259335, 6) (555719, 6)
0.00578884454470087 0.005787880540613492 0.0038598644278853163


In [9]:
X_tr, y_tr = X_train_sub, y_train_sub

In [10]:
for name, X, y in [("train", X_tr, y_tr), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print(f"{name:5s}  rows={len(X):>9,}  fraud rate={y.mean():.4%}")
print("features:", list(X_tr.columns))

train  rows=1,037,340  fraud rate=0.5789%
val    rows=  259,335  fraud rate=0.5788%
test   rows=  555,719  fraud rate=0.3860%
features: ['amt_log', 'category_risk', 'is_night', 'velocity_log', 'age', 'gender_flag']


In [11]:
import numpy as np
from sklearn.metrics import average_precision_score, f1_score, fbeta_score, matthews_corrcoef

def tune_and_evaluate(model, name):
    val_proba = model.predict_proba(X_val)[:, 1]
    thresholds = np.arange(0.01, 1.00, 0.01)
    f2_scores = [fbeta_score(y_val, (val_proba >= t).astype(int), beta=2) for t in thresholds]
    best_t = thresholds[int(np.argmax(f2_scores))]

    test_proba = model.predict_proba(X_test)[:, 1]
    test_pred = (test_proba >= best_t).astype(int)
    result = {
        "Model": name,
        "Threshold": round(best_t, 2),
        "Val PR-AUC": round(average_precision_score(y_val, val_proba), 4),
        "PR-AUC": round(average_precision_score(y_test, test_proba), 4),
        "F1": round(f1_score(y_test, test_pred), 4),
        "F2": round(fbeta_score(y_test, test_pred, beta=2), 4),
        "MCC": round(matthews_corrcoef(y_test, test_pred), 4),
    }
    print(result)
    return result

results = []

In [12]:
from imblearn.ensemble import BalancedRandomForestClassifier

brf = BalancedRandomForestClassifier(
    n_estimators=100,
    sampling_strategy="all",   # balance every tree's sample
    replacement=True,
    bootstrap=False,           # the original Balanced RF settings; also silences version warnings
    random_state=42,
    n_jobs=-1,
)
brf.fit(X_tr, y_tr)
results.append(tune_and_evaluate(brf, "Balanced RF"))

{'Model': 'Balanced RF', 'Threshold': np.float64(0.87), 'Val PR-AUC': np.float64(0.7972), 'PR-AUC': np.float64(0.7207), 'F1': 0.6326, 'F2': 0.7513, 'MCC': np.float64(0.6541)}


In [13]:
from catboost import CatBoostClassifier

cb_control = CatBoostClassifier(
    iterations=500,
    random_seed=42,
    verbose=0,
    thread_count=-1,
)
cb_control.fit(X_tr, y_tr)
results.append(tune_and_evaluate(cb_control, "CatBoost Control"))

{'Model': 'CatBoost Control', 'Threshold': np.float64(0.12), 'Val PR-AUC': np.float64(0.8744), 'PR-AUC': np.float64(0.8057), 'F1': 0.661, 'F2': 0.7632, 'MCC': np.float64(0.6766)}


In [14]:
neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
spw = neg / pos
print("scale_pos_weight:", round(spw, 2))

cb_weighted = CatBoostClassifier(
    iterations=500,
    scale_pos_weight=spw,
    random_seed=42,
    verbose=0,
    thread_count=-1,
)
cb_weighted.fit(X_tr, y_tr)
results.append(tune_and_evaluate(cb_weighted, "CatBoost Weighted"))

scale_pos_weight: 171.75
{'Model': 'CatBoost Weighted', 'Threshold': np.float64(0.91), 'Val PR-AUC': np.float64(0.8663), 'PR-AUC': np.float64(0.784), 'F1': 0.6421, 'F2': 0.754, 'MCC': np.float64(0.6611)}


In [15]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)
results.append(tune_and_evaluate(rf, "Random Forest Control"))

xgb_w = XGBClassifier(n_estimators=100, eval_metric="logloss",
                      scale_pos_weight=spw, random_state=42)
xgb_w.fit(X_tr, y_tr)
results.append(tune_and_evaluate(xgb_w, "XGBoost Weighted"))

{'Model': 'Random Forest Control', 'Threshold': np.float64(0.15), 'Val PR-AUC': np.float64(0.8663), 'PR-AUC': np.float64(0.807), 'F1': 0.6957, 'F2': 0.776, 'MCC': np.float64(0.705)}
{'Model': 'XGBoost Weighted', 'Threshold': np.float64(0.93), 'Val PR-AUC': np.float64(0.8716), 'PR-AUC': np.float64(0.7954), 'F1': 0.6668, 'F2': 0.7578, 'MCC': np.float64(0.6791)}


In [16]:
comparison = pd.DataFrame(results).sort_values("PR-AUC", ascending=False).reset_index(drop=True)
comparison

,Model,Threshold,Val PR-AUC,PR-AUC,F1,F2,MCC
0,Random Forest Control,0.15,0.8663,0.8070,0.6957,0.7760,0.7050
1,CatBoost Control,0.12,0.8744,0.8057,0.6610,0.7632,0.6766
2,XGBoost Weighted,0.93,0.8716,0.7954,0.6668,0.7578,0.6791
3,CatBoost Weighted,0.91,0.8663,0.7840,0.6421,0.7540,0.6611
4,Balanced RF,0.87,0.7972,0.7207,0.6326,0.7513,0.6541


In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

rf_settings = [
    {"max_features": "sqrt", "min_samples_leaf": 1},   # default, for reference
    {"max_features": 3,      "min_samples_leaf": 1},
    {"max_features": 4,      "min_samples_leaf": 1},
    {"max_features": "sqrt", "min_samples_leaf": 5},
    {"max_features": 3,      "min_samples_leaf": 5},
    {"max_features": 3,      "min_samples_leaf": 10},
]

rf_tuning = []
for s in rf_settings:
    m = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, **s)
    m.fit(X_tr, y_tr)
    val_pr = average_precision_score(y_val, m.predict_proba(X_val)[:, 1])
    rf_tuning.append({**s, "Val PR-AUC": round(val_pr, 4)})
    print(s, "->", round(val_pr, 4))

pd.DataFrame(rf_tuning).sort_values("Val PR-AUC", ascending=False)

{'max_features': 'sqrt', 'min_samples_leaf': 1} -> 0.8713
{'max_features': 3, 'min_samples_leaf': 1} -> 0.8779
{'max_features': 4, 'min_samples_leaf': 1} -> 0.8777
{'max_features': 'sqrt', 'min_samples_leaf': 5} -> 0.877
{'max_features': 3, 'min_samples_leaf': 5} -> 0.8837
{'max_features': 3, 'min_samples_leaf': 10} -> 0.8805


,max_features,min_samples_leaf,Val PR-AUC
4,3,5,0.8837
5,3,10,0.8805
1,3,1,0.8779
2,4,1,0.8777
3,sqrt,5,0.8770
0,sqrt,1,0.8713


In [18]:
from catboost import CatBoostClassifier

cb_tuning = []
for depth in [6, 8]:
    for lr in [0.03, 0.1]:
        m = CatBoostClassifier(
            iterations=2000, depth=depth, learning_rate=lr,
            eval_metric="PRAUC", early_stopping_rounds=100,
            random_seed=42, verbose=0, thread_count=-1,
        )
        m.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        val_pr = average_precision_score(y_val, m.predict_proba(X_val)[:, 1])
        cb_tuning.append({"depth": depth, "learning_rate": lr,
                          "best_iteration": m.get_best_iteration(),
                          "Val PR-AUC": round(val_pr, 4)})
        print(cb_tuning[-1])

pd.DataFrame(cb_tuning).sort_values("Val PR-AUC", ascending=False)

{'depth': 6, 'learning_rate': 0.03, 'best_iteration': 1960, 'Val PR-AUC': np.float64(0.8802)}
{'depth': 6, 'learning_rate': 0.1, 'best_iteration': 790, 'Val PR-AUC': np.float64(0.8819)}
{'depth': 8, 'learning_rate': 0.03, 'best_iteration': 1623, 'Val PR-AUC': np.float64(0.8817)}
{'depth': 8, 'learning_rate': 0.1, 'best_iteration': 430, 'Val PR-AUC': np.float64(0.882)}


,depth,learning_rate,best_iteration,Val PR-AUC
3,8,0.10,430,0.8820
1,6,0.10,790,0.8819
2,8,0.03,1623,0.8817
0,6,0.03,1960,0.8802


In [19]:
rf_tuned = RandomForestClassifier(n_estimators=200, max_features=3, min_samples_leaf=5,
                                  random_state=42, n_jobs=-1)
cb_tuned = CatBoostClassifier(iterations=2000, depth=8, learning_rate=0.1,
                              eval_metric="PRAUC", early_stopping_rounds=100,
                              random_seed=42, verbose=0, thread_count=-1)

In [20]:
print(len(rf_tuning), len(cb_tuning))

6 4


In [21]:
best_rf = pd.DataFrame(rf_tuning).sort_values("Val PR-AUC", ascending=False).iloc[0]
best_cb = pd.DataFrame(cb_tuning).sort_values("Val PR-AUC", ascending=False).iloc[0]
print("Best RF settings:", best_rf.to_dict())
print("Best CatBoost settings:", best_cb.to_dict())

mf = best_rf["max_features"]
rf_tuned = RandomForestClassifier(
    n_estimators=200,
    max_features=int(mf) if str(mf).isdigit() else mf,
    min_samples_leaf=int(best_rf["min_samples_leaf"]),
    random_state=42, n_jobs=-1,
)
rf_tuned.fit(X_tr, y_tr)

cb_tuned = CatBoostClassifier(
    iterations=2000, depth=int(best_cb["depth"]),
    learning_rate=float(best_cb["learning_rate"]),
    eval_metric="PRAUC", early_stopping_rounds=100,
    random_seed=42, verbose=0, thread_count=-1,
)
cb_tuned.fit(X_tr, y_tr, eval_set=(X_val, y_val))

final = [tune_and_evaluate(rf_tuned, "Random Forest Tuned"),
         tune_and_evaluate(cb_tuned, "CatBoost Tuned")]
final_table = pd.DataFrame(final).sort_values("Val PR-AUC", ascending=False).reset_index(drop=True)
print("\nWinner chosen by validation PR-AUC:", final_table.loc[0, "Model"])
final_table

Best RF settings: {'max_features': 3, 'min_samples_leaf': 5, 'Val PR-AUC': 0.8837}
Best CatBoost settings: {'depth': 8.0, 'learning_rate': 0.1, 'best_iteration': 430.0, 'Val PR-AUC': 0.882}
{'Model': 'Random Forest Tuned', 'Threshold': np.float64(0.14), 'Val PR-AUC': np.float64(0.8837), 'PR-AUC': np.float64(0.8326), 'F1': 0.6887, 'F2': 0.7813, 'MCC': np.float64(0.7012)}
{'Model': 'CatBoost Tuned', 'Threshold': np.float64(0.09), 'Val PR-AUC': np.float64(0.882), 'PR-AUC': np.float64(0.8224), 'F1': 0.6331, 'F2': 0.7632, 'MCC': np.float64(0.6587)}

Winner chosen by validation PR-AUC: Random Forest Tuned


,Model,Threshold,Val PR-AUC,PR-AUC,F1,F2,MCC
0,Random Forest Tuned,0.14,0.8837,0.8326,0.6887,0.7813,0.7012
1,CatBoost Tuned,0.09,0.8820,0.8224,0.6331,0.7632,0.6587
